# Hybrid Dataset Validation

Validation checks for the 5K+ Indonesian profile-job pair dataset.
Ensures data quality, format correctness, and suitability for SBERT fine-tuning.

In [ ]:
import json
from pathlib import Path
from collections import Counter
import pandas as pd

DATA_PATH = Path('../services/sbert/training/data/indonesian_profile_job_pairs_hybrid.jsonl')

## 1. Schema Validation

In [ ]:
required_fields = [
    'pair_id', 'pair_kind', 'profile_id', 'job_id',
    'profile_text', 'job_text', 'profile_skills', 'job_skills',
    'matched_skills', 'label', 'source_event', 'source_label',
    'provenance', 'split'
]

records = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line))

missing_counts = Counter()
for r in records:
    for field in required_fields:
        if field not in r:
            missing_counts[field] += 1

if missing_counts:
    print("Missing fields found:")
    for field, count in missing_counts.most_common():
        print(f"  {field}: {count} records")
else:
    print("All required fields present in every record.")

print(f"\nTotal records: {len(records)}")

## 2. Positive Pair Hard Negative Check

In [ ]:
positive_with_hn = sum(1 for r in records
                       if r['pair_kind'] == 'positive' and 'hard_negative_job_id' in r)
positive_total = sum(1 for r in records if r['pair_kind'] == 'positive')

print(f"Positive pairs with hard_negative: {positive_with_hn}/{positive_total}")
print(f"Coverage: {positive_with_hn/positive_total*100:.1f}%")

## 3. Label Consistency

In [ ]:
label_issues = []
for r in records:
    if r['pair_kind'] == 'positive' and r['label'] != 1.0:
        label_issues.append(f"{r['pair_id']}: positive but label={r['label']}")
    if r['pair_kind'] == 'negative' and r['label'] != 0.0:
        label_issues.append(f"{r['pair_id']}: negative but label={r['label']}")

if label_issues:
    print(f"Label inconsistencies found: {len(label_issues)}")
    for issue in label_issues[:5]:
        print(f"  {issue}")
else:
    print("All labels consistent with pair_kind.")

## 4. Duplicate Check

In [ ]:
pair_ids = [r['pair_id'] for r in records]
duplicates = {pid: count for pid, count in Counter(pair_ids).items() if count > 1}

if duplicates:
    print(f"Duplicate pair_ids found: {len(duplicates)}")
    for pid, count in list(duplicates.items())[:5]:
        print(f"  {pid}: {count} occurrences")
else:
    print("No duplicate pair_ids found.")

## 5. Empty Text Check

In [ ]:
empty_text_issues = []
for r in records:
    if not str(r.get('profile_text', '')).strip():
        empty_text_issues.append(f"{r['pair_id']}: empty profile_text")
    if not str(r.get('job_text', '')).strip():
        empty_text_issues.append(f"{r['pair_id']}: empty job_text")

if empty_text_issues:
    print(f"Empty text issues: {len(empty_text_issues)}")
    for issue in empty_text_issues[:5]:
        print(f"  {issue}")
else:
    print("No empty text fields found.")

## 6. Split Distribution Check

In [ ]:
splits = Counter(r['split'] for r in records)
print(f"Split distribution: {dict(splits)}")

total = len(records)
for split, count in splits.items():
    pct = count / total * 100
    print(f"  {split}: {count} ({pct:.1f}%)")

# Check positive/negative balance per split
df = pd.DataFrame(records)
print("\nPair kind by split:")
print(df.groupby(['split', 'pair_kind']).size().unstack(fill_value=0))

## 7. Skill Overlap Analysis (Hard Negative Quality)

In [ ]:
positive_records = [r for r in records if r['pair_kind'] == 'positive']

pos_overlaps = []
neg_overlaps = []

for r in positive_records:
    profile_skills = set(s.lower() for s in r['profile_skills'])
    pos_skills = set(s.lower() for s in r['job_skills'])
    pos_overlap = len(profile_skills & pos_skills)
    pos_overlaps.append(pos_overlap)

    if 'hard_negative_skills' in r:
        neg_skills = set(s.lower() for s in r['hard_negative_skills'])
        neg_overlap = len(profile_skills & neg_skills)
        neg_overlaps.append(neg_overlap)

print(f"Positive overlap - mean: {sum(pos_overlaps)/len(pos_overlaps):.2f}, max: {max(pos_overlaps)}")
if neg_overlaps:
    print(f"Hard negative overlap - mean: {sum(neg_overlaps)/len(neg_overlaps):.2f}, max: {max(neg_overlaps)}")

import matplotlib.pyplot as plt
plt.hist([pos_overlaps, neg_overlaps], bins=range(0, max(max(pos_overlaps), max(neg_overlaps))+2),
         label=['Positive', 'Hard Negative'], alpha=0.7)
plt.xlabel('Skill Overlap Count')
plt.ylabel('Frequency')
plt.title('Skill Overlap: Positive vs Hard Negative')
plt.legend()
plt.show()

## 8. Final Validation Report

In [ ]:
report = {
    'total_records': len(records),
    'positive_pairs': sum(1 for r in records if r['pair_kind'] == 'positive'),
    'negative_pairs': sum(1 for r in records if r['pair_kind'] == 'negative'),
    'schema_valid': len(missing_counts) == 0,
    'labels_consistent': len(label_issues) == 0,
    'no_duplicates': len(duplicates) == 0,
    'no_empty_text': len(empty_text_issues) == 0,
    'hard_negative_coverage': positive_with_hn / positive_total * 100 if positive_total > 0 else 0,
}

print("=== VALIDATION REPORT ===")
for key, value in report.items():
    if isinstance(value, bool):
        status = "PASS" if value else "FAIL"
        print(f"{key}: {status}")
    elif isinstance(value, float):
        print(f"{key}: {value:.1f}%")
    else:
        print(f"{key}: {value}")

all_pass = all(v for k, v in report.items() if isinstance(v, bool))
print(f"\nOverall: {'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}")